# 공통(Common) 메트릭 데모 — SDK & API

이 노트북은 모든 에이전트 유형에 공통으로 적용되는 **공통 메트릭**을 두 가지 방식으로 계산한다.

- **SDK**: 메트릭 클래스를 직접 임포트해 `EvalContext`에 대해 계산한다.
- **API**: FastAPI 메트릭 서버의 엔드포인트에 HTTP로 요청해 계산한다.

대상 메트릭: `llm_judge`, `bertscore`, `p95_latency`, `token_usage`

## 사전 준비

```bash
uv sync --extra server --extra t2s --extra bertscore   # bertscore는 torch를 설치한다(용량 큼)
```

이 프로젝트의 가상환경 커널로 이 노트북을 실행한다.

> 참고: `llm_judge`는 아래에서 설정하는 **실제 LLM**으로, `bertscore`는 **실제 mBERT 모델**
> (최초 1회 약 700MB 다운로드)로 계산한다. judge는 temperature 0으로 호출하지만 LLM 특성상
> 실행마다 점수가 조금 달라질 수 있다. 이 노트북 전체 실행 시 judge 호출은 6회다
> (항목 3 × SDK/API 두 파트) — 비용은 소액이다.

## 0. 데이터셋

하드코딩된 간단한 질의·응답 데이터셋이다. 각 항목은 질문(`input`), 에이전트 응답(`output`), 정답(`expected`), 그리고 성능 메타데이터(`latency_ms`, `tokens`)를 갖는다.

In [ ]:
DATASET = [
    {"input": "프랑스의 수도는 어디인가요?", "output": "프랑스의 수도는 파리입니다.", "expected": "파리", "latency_ms": 820.0, "tokens": 45},
    {"input": "물의 화학식은 무엇인가요?", "output": "물의 화학식은 H2O입니다.", "expected": "H2O", "latency_ms": 1250.0, "tokens": 60},
    {"input": "지구에서 가장 큰 대양은?", "output": "가장 큰 대양은 태평양입니다.", "expected": "태평양", "latency_ms": 2100.0, "tokens": 52},
]
for i, row in enumerate(DATASET):
    print(i, row)

## 실제 judge LLM 설정

judge 기반 메트릭은 **실제 LLM**으로 채점한다. 공급자는 코드 인자가 아니라 환경변수
`AGENT_EVAL_JUDGE_PROVIDER` 하나로 전환하며(`anthropic` ↔ `openai`), 키는 레포 루트의
`.env` 파일에 넣는다(`.gitignore`에 의해 커밋되지 않음):

```
# .env 예시 — 둘 중 하나(또는 둘 다) 설정
AGENT_EVAL_JUDGE_PROVIDER=anthropic
ANTHROPIC_API_KEY=sk-ant-...

# AGENT_EVAL_JUDGE_PROVIDER=openai
# OPENAI_API_KEY=sk-...
```

이 규약(공급자 프리셋 포함)은 노트북 전용이 아니라 **서버 패키지(`agent_eval.server.judge`)의
공식 규약**이다 — 아래 셀은 서버가 시작할 때 부르는 바로 그 리졸버(`resolve_judge_from_env`)를
그대로 사용하므로, SDK 파트와 API 서버는 항상 같은 judge를 쓴다. 노트북 밖에서
`uv run agent-eval-serve`로 서버를 단독 실행해도 같은 `./.env`를 자동으로 읽는다.

공급자/모델을 바꾼 뒤에는 커널을 재시작한다(API 서버는 시작 시점의 설정을 계속 쓴다).
사내 게이트웨이·vLLM 등 임의의 OpenAI 호환 서버는 `AGENT_EVAL_JUDGE_BASE_URL`/`_MODEL`/
`_API_KEY`를 직접 지정하면 되고(프리셋보다 우선), Anthropic SDK를 직접 쓰는 팩토리 방식은
`AGENT_EVAL_JUDGE_FACTORY=examples/judges.py:claude_judge` 처럼 지정한다(최우선).

In [ ]:
# judge 설정은 서버와 완전히 같은 규약을 쓴다(agent_eval.server.judge — 서버 시작 시 부르는
# 바로 그 리졸버). 환경변수 하나로 실제 judge LLM 공급자를 전환한다:
#   AGENT_EVAL_JUDGE_PROVIDER=anthropic  →  Claude (ANTHROPIC_API_KEY 필요)
#   AGENT_EVAL_JUDGE_PROVIDER=openai     →  GPT    (OPENAI_API_KEY 필요)
import os
from pathlib import Path

from agent_eval.server.judge import load_env_file, resolve_judge_from_env

# 노트북 폴더에서 실행하든 레포 루트에서 실행하든 .env를 찾도록 둘 다 시도한다
# (이미 설정된 환경변수가 항상 우선한다).
load_env_file(Path.cwd() / ".env")
load_env_file(Path.cwd().parent / ".env")

# 실제 LLM 없이 스텁으로 조용히 떨어지지 않도록 미리 막고, 한국어로 안내한다.
if not (
    os.environ.get("AGENT_EVAL_JUDGE_FACTORY")
    or os.environ.get("AGENT_EVAL_JUDGE_BASE_URL")
    or os.environ.get("AGENT_EVAL_JUDGE_PROVIDER")
):
    raise RuntimeError(
        "실제 LLM judge 설정이 없다. 레포 루트의 .env(또는 셸)에 다음 중 하나를 설정한다:\n"
        "  AGENT_EVAL_JUDGE_PROVIDER=anthropic  (그리고 ANTHROPIC_API_KEY=...)\n"
        "  AGENT_EVAL_JUDGE_PROVIDER=openai     (그리고 OPENAI_API_KEY=...)\n"
        "  또는 AGENT_EVAL_JUDGE_BASE_URL / _MODEL / _API_KEY 직접 지정"
    )

resolved = resolve_judge_from_env()  # API 서버가 시작할 때 부르는 바로 그 리졸버
judge = resolved.backend             # SDK 파트에서 그대로 쓸 실제 LLM judge
print(f"judge: kind={resolved.kind}  detail={resolved.detail}")

---
# Part 1. SDK

메트릭 클래스를 직접 임포트해 계산한다. judge는 위 설정 셀에서 만든 `judge`(서버와 같은 리졸버의 산출물)를 그대로 쓴다 — 내부적으로는 OpenAI 호환 `/chat/completions`를 부르는 `complete(prompt) -> str` 콜러블(temperature 0) 위에서 프레임워크의 `LLMJudge`가 프롬프트 렌더링과 `SCORE:`/`REASON:` 파싱을 담당한다.

In [ ]:
from agent_eval.core.contracts import EvalContext, MetaKey
from agent_eval.metrics.common import BertScore, LLMJudgeMetric, P95Latency, TokenUsage

# 데이터셋 전체를 러너로 집계해 대표값과 95% 신뢰구간(CI)을 구하는 헬퍼.
# 각 항목은 '한 번만' 채점하고 그 결과를 러너에 재생(replay)해 집계한다 —
# judge 메트릭이 같은 항목으로 LLM을 두 번 호출하지 않도록, API 서버와 동일한 방식이다.
from types import SimpleNamespace

from agent_eval.core.gate import GatePolicy
from agent_eval.core.suite import Suite
from agent_eval.offline.runner import evaluate


def aggregate(metric, results, ctxs):
    replay = iter(results)
    shim = SimpleNamespace(  # 집계 속성만 흉내 내고, score()는 이미 계산한 결과를 돌려준다
        name=metric.name,
        requires=frozenset(),
        cost_class=metric.cost_class,
        aggregation=metric.aggregation,
        higher_is_better=metric.higher_is_better,
        unit_interval=getattr(metric, "unit_interval", True),
        score=lambda ctx: next(replay),
    )
    return evaluate(Suite("demo", metric.name, [shim], GatePolicy()), ctxs).aggregates[0]


def run_sdk(metric, ctxs):
    """각 항목을 개별 채점해 출력하고, 같은 결과로 데이터셋 집계를 출력한다."""
    print(f"[SDK] {metric.name}")
    results = [metric.score(ctx) for ctx in ctxs]
    for i, r in enumerate(results):
        print(f"  #{i}: score={r.score:.3f}  passed={r.passed}  error={r.error}")
        reason = (r.detail or {}).get("reason", "")
        if reason:
            print(f"      └ 판정 이유: {str(reason)[:110]}")
    agg = aggregate(metric, results, ctxs)
    print(f"  ▶ 집계 value={agg.value:.3f}  95% CI=[{agg.ci_low:.3f}, {agg.ci_high:.3f}]  n={agg.n}")

# 공통 메트릭은 최종 응답(output)과 메타데이터를 읽는다.
contexts = [
    EvalContext(
        input=row["input"],
        output=row["output"],
        expected=row["expected"],
        metadata={MetaKey.LATENCY_MS: row["latency_ms"], MetaKey.TOKENS: row["tokens"]},
    )
    for row in DATASET
]

# judge는 위 설정 셀에서 이미 만들었다(서버와 같은 리졸버의 산출물).
print("준비 완료:", len(contexts), "개 컨텍스트, judge =", resolved.detail)

## 1-1. `llm_judge` — 응답 품질 심판

질문(`input`)과 응답(`output`)을, 있다면 정답(`expected`)과 함께 **실제 LLM 심판**에게 넘겨 주관적 품질 점수(0~1)를 받는다. `판정 이유`는 심판이 돌려준 `REASON:` 줄이다.

In [ ]:
run_sdk(LLMJudgeMetric(judge), contexts)

## 1-2. `bertscore` — 정답과의 의미적 유사도

응답(`output`)과 정답(`expected`)의 토큰 임베딩 유사도(BERTScore F1)를 **실제 mBERT 모델**로 계산한다. 한국어(`lang="ko"`)는 `bert-base-multilingual-cased`를 쓰는데, 이 조합에는 rescale 베이스라인 파일이 없어 `rescale_with_baseline=False`로 원점수(raw F1)를 그대로 본다 — 원점수는 대체로 0.6~1.0 사이에 몰린다. 항목마다 모델을 로드하므로 수십 초 걸릴 수 있다.

In [ ]:
# 실제 BERTScore: 첫 호출 시 모델 가중치를 내려받아 캐시한다(~700MB, 최초 1회).
run_sdk(BertScore(lang="ko", rescale_with_baseline=False), contexts)

## 1-3. `p95_latency` — 꼬리(p95) 지연시간

각 항목은 `metadata['latency_ms']`를 그대로 돌려주고, 집계에서 95백분위수(p95)를 구한다. **낮을수록 좋다.**

In [ ]:
run_sdk(P95Latency(), contexts)

## 1-4. `token_usage` — 평균 토큰 사용량

`metadata['tokens']`의 평균. **낮을수록 좋다.**

In [ ]:
run_sdk(TokenUsage(), contexts)

---
# Part 2. API

동일한 데이터셋을 이번에는 HTTP 엔드포인트로 계산한다. 먼저 서버를 띄운다 — health의 `judge` 항목이 위에서 설정한 실제 모델을 가리키는지 확인한다.

In [ ]:
# API 파트: 백그라운드 스레드에서 실제 FastAPI 서버를 띄우고, httpx로 진짜 HTTP 요청을 보낸다.
# 서버는 위 설정 셀이 만든 AGENT_EVAL_JUDGE_* 환경변수를 읽어 SDK 파트와 '같은' 실제 judge를
# 쓴다 — 아래 health 출력에서 judge.kind = openai_compatible, detail = 모델명으로 확인할 수 있다.
import threading
import time

import httpx
import uvicorn

from agent_eval.server.app import create_app

PORT = 8077
BASE_URL = f"http://127.0.0.1:{PORT}"

if "server" not in globals():
    server = uvicorn.Server(uvicorn.Config(create_app(), host="127.0.0.1", port=PORT, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()
    while not server.started:
        time.sleep(0.1)
print("API 서버 준비 완료:", BASE_URL)
print("health:", httpx.get(f"{BASE_URL}/health").json())

def call_api(path, contexts, params=None):
    """엔드포인트에 contexts를 POST하고, 개별 결과와 집계를 출력한다."""
    payload = {"contexts": contexts}
    if params:
        payload["params"] = params
    resp = httpx.post(BASE_URL + path, json=payload, timeout=120)
    print(f"[API] POST {path} → {resp.status_code}")
    data = resp.json()
    if resp.status_code != 200:
        print("  오류:", data.get("detail"))
        return data
    for i, item in enumerate(data["results"]):
        print(f"  #{i}: score={item['score']:.3f}  passed={item['passed']}  error={item['error']}")
        reason = (item.get("detail") or {}).get("reason", "")
        if reason:
            print(f"      └ 판정 이유: {str(reason)[:110]}")
    agg = data["aggregate"]
    print(f"  ▶ 집계 value={agg['value']:.3f}  95% CI=[{agg['ci_low']:.3f}, {agg['ci_high']:.3f}]  n={agg['n']}")
    return data

## 2-1. `POST /common/llm_judge`

각 컨텍스트는 `input`, `output`(선택적으로 `expected`)를 담는다. 같은 judge 설정이므로 SDK 1-1과 사실상 같은 점수가 나온다(LLM 특성상 미세 차이는 가능).

In [ ]:
ctx = [{"input": r["input"], "output": r["output"], "expected": r["expected"]} for r in DATASET]
call_api("/common/llm_judge", ctx)

## 2-2. `POST /common/bertscore`

서버 venv에 `bert-score`가 설치되어 있으므로 실제 점수를 돌려준다(미설치 서버라면 **501** + 설치 힌트). BERTScore는 결정적이라 SDK 1-2와 정확히 일치한다. `params`로 언어/rescale 설정을 넘긴다.

In [ ]:
ctx = [{"output": r["output"], "expected": r["expected"]} for r in DATASET]
call_api("/common/bertscore", ctx, params={"lang": "ko", "rescale_with_baseline": False})

## 2-3. `p95_latency` · `token_usage` — API 엔드포인트 없음

이 두 메트릭은 입력/출력을 채점하지 않고 하니스가 채워준 메타데이터만 읽기 때문에 **서버 엔드포인트가 없다**. 위 SDK 1-3 / 1-4처럼 라이브러리로 계산한다.